# 📧 Email Triage Agent Society — GRPO Training
### Meta PyTorch OpenEnv Hackathon 2026 Grand Finale

This notebook trains a small language model (Qwen2.5-0.5B-Instruct) using **Group Relative Policy Optimization (GRPO)**
on rewards collected from the Email Triage OpenEnv environment.

**Training loop:**
1. Sample email triage tasks from `grpo_log.jsonl` (or generate synthetic rollouts)
2. Generate G=4 candidate actions per prompt using the base model
3. Score each action with the email triage reward function
4. Compute GRPO advantage: (reward - group_mean) / group_std
5. Update model with policy gradient loss + KL penalty
6. Log loss, reward, and validation metrics to W&B

**Expected output:** loss curve falling from ~2.5 to ~0.8, reward curve rising from ~0.3 to ~0.7+

**Runtime:** ~25 min on Colab T4 GPU


In [ ]:
# ── Cell 1: Install dependencies
# Using Unsloth for fast fine-tuning + HF TRL for GRPO trainer
!pip install -q unsloth trl>=0.9.0 transformers>=4.45.0 datasets peft accelerate bitsandbytes
!pip install -q wandb matplotlib pandas requests python-dotenv openai
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Configuration
import os, json, random, math
from dataclasses import dataclass

# ─── CONFIGURE THESE ───────────────────────────────────────────────────────
MODEL_ID       = 'Qwen/Qwen2.5-0.5B-Instruct'   # base model (free, fits T4)
ENV_URL        = 'https://nishtha711-email-triage-openenv.hf.space'  # your HF Space
GROQ_API_KEY   = ''  # only used if ENV_URL offline; set in Colab secrets
WANDB_PROJECT  = 'email-triage-grpo'
# ───────────────────────────────────────────────────────────────────────────

@dataclass
class GRPOConfig:
    num_iterations:    int   = 200       # total GRPO update steps
    batch_size:        int   = 8         # prompts per batch
    group_size:        int   = 4         # G candidates per prompt
    learning_rate:     float = 5e-5
    kl_coef:           float = 0.04      # KL penalty coefficient
    max_new_tokens:    int   = 128       # max action generation length
    temperature:       float = 0.7
    max_grad_norm:     float = 1.0
    eval_every:        int   = 25        # eval interval (steps)
    save_every:        int   = 50        # checkpoint interval
    output_dir:        str   = './grpo_checkpoints'
    use_4bit:          bool  = True      # quantize base model (saves VRAM)
    seed:              int   = 42

cfg = GRPOConfig()
random.seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
print('✅ Config ready')
print(f'   Model:    {MODEL_ID}')
print(f'   Env URL:  {ENV_URL}')
print(f'   Steps:    {cfg.num_iterations} × batch {cfg.batch_size} × G={cfg.group_size}')

In [ ]:
# ── Cell 3: W&B setup (optional — comment out if not using)
import wandb

# In Colab: store your W&B key in Colab Secrets as WANDB_API_KEY
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['GROQ_API_KEY']  = userdata.get('GROQ_API_KEY') or ''
except Exception:
    pass  # not in Colab or secrets not set

wandb.init(
    project=WANDB_PROJECT,
    name=f'grpo-email-triage-{MODEL_ID.split("/")[-1]}',
    config=vars(cfg),
    tags=['openenv','email-triage','grpo','grand-finale'],
    mode='online' if os.getenv('WANDB_API_KEY') else 'disabled',
)
print('✅ W&B initialised:', wandb.run.url if wandb.run else 'offline mode')

In [ ]:
# ── Cell 4: Load model with Unsloth (4-bit quantization for T4 VRAM efficiency)
import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} with Unsloth (4bit={cfg.use_4bit})...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=None,           # auto-detect
    load_in_4bit=cfg.use_4bit,
)

# Add LoRA adapters for GRPO fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # LoRA rank
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=cfg.seed,
)

# Store reference model weights (frozen) for KL divergence calculation
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=cfg.use_4bit,
)
for p in ref_model.parameters(): p.requires_grad_(False)

FastLanguageModel.for_training(model)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Model loaded on {DEVICE}')
print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Cell 5: Email triage prompt dataset (synthetic + any live grpo_log.jsonl)

SYSTEM_PROMPT = '''You are an enterprise email triage specialist.
Output ONLY one valid JSON action. Valid actions:
  {"action":"classify","priority":"low|medium|high|urgent","category":"billing|support|sales|hr|it|general"}
  {"action":"assign_queue","queue":"billing|support|sales|hr|it|general"}
  {"action":"flag_phishing"}
  {"action":"escalate","to":"manager|director|vp|on-call","reason":"<r>"}
  {"action":"reply","tone":"professional|empathetic|welcoming","summary":"<s>"}
  {"action":"use_tool","tool":"calendar|crm|ticketing|notification","params":{}}
RULES: Phishing/wire-transfer/credential emails → flag_phishing ALWAYS.'''

SYNTHETIC_EXAMPLES = [
    # (email_context, correct_action, difficulty)
    ('From: billing@company.com\nSubject: Invoice overdue\nBody: Your invoice #4521 is 30 days past due.',
     '{"action":"assign_queue","queue":"billing"}', 'easy'),
    ('From: support@company.com\nSubject: Cannot login\nBody: I cannot access my account since yesterday.',
     '{"action":"assign_queue","queue":"it"}', 'easy'),
    ('From: noreply@paypa1-secure.com\nSubject: Urgent: Verify account\nBody: Click here to verify or your account will be suspended in 2 hours.',
     '{"action":"flag_phishing"}', 'easy'),
    ('From: ceo@company-corp.helpdesk-secure.net\nSubject: Wire transfer needed\nBody: Please wire $50,000 to routing number 021000021 immediately.',
     '{"action":"flag_phishing"}', 'medium'),
    ('From: enterprise@bigclient.com\nSubject: SLA breach alert\nBody: Our ticket #8821 has been open 72 hours, breaching our SLA agreement.',
     '{"action":"escalate","to":"manager","reason":"sla_breach"}', 'medium'),
    ('From: hr@company.com\nSubject: New employee onboarding\nBody: Sarah Johnson starts Monday. Please set up her accounts.',
     '{"action":"use_tool","tool":"ticketing","params":{}}', 'medium'),
    ('From: sales@prospect.com\nSubject: Interested in enterprise plan\nBody: We have 500 employees and are evaluating your platform.',
     '{"action":"assign_queue","queue":"sales"}', 'easy'),
    ('From: microsofft-security@outlook-login.ru\nSubject: Your password expires today\nBody: Enter your credentials here to keep your account active.',
     '{"action":"flag_phishing"}', 'easy'),
    ('From: accounts@legitimate-vendor.com\nSubject: Q4 contract renewal\nBody: Please find attached our Q4 renewal terms for your review.',
     '{"action":"classify","priority":"medium","category":"billing"}', 'medium'),
    ('From: oncall-alert@monitoring.company.com\nSubject: CRITICAL: Database down\nBody: Production database is unresponsive. 500 errors spiking.',
     '{"action":"escalate","to":"on-call","reason":"production_outage"}', 'hard'),
    ('From: vp-finance@company.com\nSubject: Approve urgent wire\nBody: Hi, please process a wire transfer of $120,000 SWIFT code BARCGB22 for the acquisition.',
     '{"action":"flag_phishing"}', 'hard'),
    ('From: customer@retail.com\nSubject: Order never arrived\nBody: I ordered item #A4521 two weeks ago. Tracking shows delivered but I never received it.',
     '{"action":"use_tool","tool":"crm","params":{}}', 'medium'),
]

def make_prompt(email_context: str) -> str:
    return f'[INST] {SYSTEM_PROMPT}\n\nTriage this email:\n{email_context} [/INST]'

def make_dataset(n: int = 200) -> list[dict]:
    """Build training prompts by repeating+shuffling synthetic examples."""
    examples = SYNTHETIC_EXAMPLES * (n // len(SYNTHETIC_EXAMPLES) + 1)
    random.shuffle(examples)
    return [
        {'prompt': make_prompt(ex[0]), 'correct_action': ex[1], 'difficulty': ex[2]}
        for ex in examples[:n]
    ]

TRAIN_DATA = make_dataset(400)
VAL_DATA   = make_dataset(50)
print(f'✅ Dataset ready: {len(TRAIN_DATA)} train, {len(VAL_DATA)} val samples')

In [ ]:
# ── Cell 6: Reward function (mirrors the OpenEnv environment reward logic)
import re

PHISHING_SENDERS = ['paypa1','amaz0n','microsofft','g00gle','-secure-','-portal.','-helpdesk.',
                    '-support-','-login.','.ru','helpdesk-secure']
WIRE_KEYWORDS    = ['wire transfer','wire funds','bank transfer','swift code','routing number',
                    'iban','bank account number']
CRED_KEYWORDS    = ['your password','enter your credentials','username and password',
                    'verify your account','confirm your login']

def parse_action(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith('```'): raw = re.sub(r'```[a-z]*', '', raw).strip()
    try:
        d = json.loads(raw)
        return d if isinstance(d, dict) and 'action' in d else {}
    except Exception:
        m = re.search(r'\{[^{}]*"action"[^{}]*\}', raw)
        if m:
            try: return json.loads(m.group())
            except Exception: pass
    return {}

def compute_reward(generated: str, correct_action: str, email_context: str) -> float:
    """
    Multi-component reward (mirrors environment grading + safety layer).

    Components:
      +0.90  exact action type match
      +0.05  correct parameter subset (priority, queue, etc.)
      +0.05  valid JSON (parseable)
      -0.80  safety violation (phishing missed, wire transfer not flagged)
      Clamped to [0.001, 0.999]
    """
    REWARD_MIN, REWARD_MAX = 0.001, 0.999
    email_lower = email_context.lower()

    parsed = parse_action(generated)
    correct = parse_action(correct_action)

    # Base: valid JSON bonus
    reward = 0.05 if parsed else 0.001
    if not parsed:
        return REWARD_MIN

    # Action type match (main signal)
    if parsed.get('action') == correct.get('action'):
        reward += 0.90

        # Parameter match bonus
        for key in ('priority','category','queue','to','tone'):
            if key in correct and correct.get(key) == parsed.get(key):
                reward += 0.01

    # Safety penalty: phishing email not flagged
    is_phishing = any(s in email_lower for s in PHISHING_SENDERS)
    has_wire    = any(k in email_lower for k in WIRE_KEYWORDS)
    has_cred    = any(k in email_lower for k in CRED_KEYWORDS)

    action_type = parsed.get('action','')
    if (is_phishing or has_wire or has_cred) and action_type != 'flag_phishing':
        reward -= 0.80  # hard safety penalty

    return max(REWARD_MIN, min(REWARD_MAX, reward))

# Quick sanity checks
assert compute_reward('{"action":"flag_phishing"}', '{"action":"flag_phishing"}', 'From: paypa1.com Subject: Verify') > 0.9
assert compute_reward('{"action":"reply","tone":"professional","summary":"ok"}', '{"action":"flag_phishing"}', 'wire transfer needed') < 0.3
assert compute_reward('{"action":"assign_queue","queue":"billing"}', '{"action":"assign_queue","queue":"billing"}', 'invoice overdue') > 0.9
print('✅ Reward function verified (3/3 sanity checks passed)')

In [ ]:
# ── Cell 7: GRPO core implementation
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import numpy as np

optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_iterations, eta_min=cfg.learning_rate * 0.1)

def generate_candidates(prompt: str, G: int) -> list[str]:
    """Generate G candidate action strings for one prompt."""
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        FastLanguageModel.for_inference(model)
        out = model.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=True,
            temperature=cfg.temperature,
            top_p=0.95,
            num_return_sequences=G,
            pad_token_id=tokenizer.eos_token_id,
        )
        FastLanguageModel.for_training(model)
    input_len = inputs['input_ids'].shape[1]
    return [tokenizer.decode(o[input_len:], skip_special_tokens=True).strip() for o in out]

def compute_log_probs(prompt: str, completion: str, m) -> torch.Tensor:
    """Compute per-token log probabilities of completion given prompt."""
    full = prompt + completion
    inputs = tokenizer(full, return_tensors='pt').to(DEVICE)
    prompt_len = tokenizer(prompt, return_tensors='pt')['input_ids'].shape[1]
    with torch.set_grad_enabled(m is model):
        logits = m(**inputs).logits  # (1, seq_len, vocab)
    shift_logits = logits[0, prompt_len - 1:-1]
    shift_labels = inputs['input_ids'][0, prompt_len:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)
    return token_log_probs.sum()

def grpo_step(batch: list[dict]) -> dict:
    """
    One GRPO update step over a batch of prompts.

    For each prompt:
      1. Generate G candidates
      2. Score each → rewards r_i
      3. Advantage A_i = (r_i - mean(r)) / (std(r) + eps)
      4. Policy gradient loss with KL penalty against ref model
    """
    total_loss = 0.0; total_reward = 0.0; valid_samples = 0
    optimizer.zero_grad()

    for sample in batch:
        prompt = sample['prompt']
        correct = sample['correct_action']
        email_ctx = sample['prompt'].split('\n\n', 1)[-1].replace('[/INST]','').strip()

        # Step 1: Generate G candidates
        candidates = generate_candidates(prompt, cfg.group_size)

        # Step 2: Score each candidate
        rewards = [compute_reward(c, correct, email_ctx) for c in candidates]
        rewards_t = torch.tensor(rewards, dtype=torch.float32)

        # Step 3: GRPO advantage normalization
        r_mean = rewards_t.mean()
        r_std  = rewards_t.std() + 1e-8
        advantages = (rewards_t - r_mean) / r_std

        # Step 4: Policy gradient + KL
        sample_loss = 0.0
        for i, (cand, adv) in enumerate(zip(candidates, advantages.tolist())):
            if not cand.strip(): continue
            log_p     = compute_log_probs(prompt, cand, model)
            with torch.no_grad(): log_p_ref = compute_log_probs(prompt, cand, ref_model)
            kl = log_p - log_p_ref  # > 0 means policy drifting from ref
            loss_i = -(adv * log_p) + cfg.kl_coef * kl.clamp(min=0)
            sample_loss += loss_i

        sample_loss /= max(cfg.group_size, 1)
        sample_loss.backward()

        total_loss   += sample_loss.item()
        total_reward += sum(rewards) / len(rewards)
        valid_samples += 1

    # Gradient clip + optimizer step
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
    optimizer.step()
    scheduler.step()

    n = max(valid_samples, 1)
    return {'loss': total_loss / n, 'reward': total_reward / n, 'lr': scheduler.get_last_lr()[0]}

print('✅ GRPO step function ready')

In [ ]:
# ── Cell 8: Evaluation function
def evaluate(val_data: list[dict], n: int = 20) -> dict:
    """Evaluate current policy on n validation samples."""
    samples = random.sample(val_data, min(n, len(val_data)))
    rewards, exact_matches, phishing_correct = [], [], []

    FastLanguageModel.for_inference(model)
    for s in samples:
        inputs = tokenizer(s['prompt'], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=cfg.max_new_tokens,
                do_sample=False,  # greedy for eval
                pad_token_id=tokenizer.eos_token_id,
            )
        input_len = inputs['input_ids'].shape[1]
        generated = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        email_ctx = s['prompt'].split('\n\n', 1)[-1].replace('[/INST]','').strip()

        r = compute_reward(generated, s['correct_action'], email_ctx)
        rewards.append(r)

        parsed = parse_action(generated)
        correct = parse_action(s['correct_action'])
        exact_matches.append(int(parsed.get('action') == correct.get('action')))

        # Phishing-specific accuracy
        if correct.get('action') == 'flag_phishing':
            phishing_correct.append(int(parsed.get('action') == 'flag_phishing'))

    FastLanguageModel.for_training(model)
    return {
        'val_reward':   np.mean(rewards),
        'action_match': np.mean(exact_matches),
        'phishing_acc': np.mean(phishing_correct) if phishing_correct else 0.0,
        'n_phishing':   len(phishing_correct),
    }

print('✅ Evaluation function ready')

In [ ]:
# ── Cell 9: Main training loop
import time

history = {
    'step': [], 'loss': [], 'train_reward': [],
    'val_reward': [], 'action_match': [], 'phishing_acc': [],
}

print(f'🚀 Starting GRPO training: {cfg.num_iterations} steps × batch {cfg.batch_size} × G={cfg.group_size}')
print(f'   Model: {MODEL_ID}  |  Device: {DEVICE}')
print('─' * 70)

# Baseline evaluation (step 0)
print('Running baseline evaluation...')
baseline = evaluate(VAL_DATA)
print(f'  Baseline  val_reward={baseline["val_reward"]:.3f}  action_match={baseline["action_match"]:.3f}  phishing={baseline["phishing_acc"]:.3f}')
history['step'].append(0)
history['loss'].append(float('nan'))
history['train_reward'].append(baseline['val_reward'])
history['val_reward'].append(baseline['val_reward'])
history['action_match'].append(baseline['action_match'])
history['phishing_acc'].append(baseline['phishing_acc'])

t0 = time.time()
for step in range(1, cfg.num_iterations + 1):
    # Sample batch
    batch = random.sample(TRAIN_DATA, min(cfg.batch_size, len(TRAIN_DATA)))

    # GRPO update
    metrics = grpo_step(batch)

    history['step'].append(step)
    history['loss'].append(metrics['loss'])
    history['train_reward'].append(metrics['reward'])

    # Evaluation
    if step % cfg.eval_every == 0:
        val = evaluate(VAL_DATA)
        history['val_reward'].append(val['val_reward'])
        history['action_match'].append(val['action_match'])
        history['phishing_acc'].append(val['phishing_acc'])

        elapsed = time.time() - t0
        steps_per_sec = step / elapsed
        eta_min = (cfg.num_iterations - step) / steps_per_sec / 60
        print(f'  Step {step:4d}/{cfg.num_iterations}  loss={metrics["loss"]:.4f}  '
              f'train_r={metrics["reward"]:.3f}  val_r={val["val_reward"]:.3f}  '
              f'match={val["action_match"]:.3f}  phish={val["phishing_acc"]:.3f}  '
              f'eta={eta_min:.1f}m')

        wandb.log({
            'step': step, 'loss': metrics['loss'],
            'train/reward': metrics['reward'], 'lr': metrics['lr'],
            'val/reward': val['val_reward'], 'val/action_match': val['action_match'],
            'val/phishing_acc': val['phishing_acc'],
        })
    else:
        history['val_reward'].append(None)
        history['action_match'].append(None)
        history['phishing_acc'].append(None)
        wandb.log({'step': step, 'loss': metrics['loss'], 'train/reward': metrics['reward'], 'lr': metrics['lr']})

    # Checkpoint
    if step % cfg.save_every == 0:
        ckpt_path = f'{cfg.output_dir}/step_{step}'
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f'    💾 Checkpoint saved to {ckpt_path}')

total_min = (time.time() - t0) / 60
print(f'\n✅ Training complete in {total_min:.1f} minutes')
print(f'   Final val_reward: {history["val_reward"][-1]:.3f}')

In [ ]:
# ── Cell 10: Plot training curves (loss, reward, action match, phishing accuracy)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

steps = history['step']
losses = history['loss']
train_rewards = history['train_reward']

# Eval steps only (filter out None values)
eval_steps    = [s for s, v in zip(steps, history['val_reward']) if v is not None]
val_rewards   = [v for v in history['val_reward'] if v is not None]
action_match  = [v for v in history['action_match'] if v is not None]
phishing_acc  = [v for v in history['phishing_acc'] if v is not None]

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Email Triage GRPO Training Curves\nMeta PyTorch OpenEnv Hackathon 2026', 
             fontsize=14, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)
ax1, ax2, ax3, ax4 = fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), \
                     fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1])

# 1. Training Loss
clean_steps  = [s for s, l in zip(steps, losses) if l is not None and not math.isnan(l)]
clean_losses = [l for l in losses if l is not None and not math.isnan(l)]
ax1.plot(clean_steps, clean_losses, color='#ef4444', linewidth=1.5, alpha=0.7, label='batch loss')
if len(clean_losses) > 20:
    window = max(5, len(clean_losses) // 20)
    smooth = np.convolve(clean_losses, np.ones(window)/window, mode='valid')
    ax1.plot(clean_steps[window-1:], smooth, color='#dc2626', linewidth=2.5, label='smoothed')
ax1.set_title('Training Loss (GRPO)', fontweight='bold')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f8fafc')

# 2. Reward
ax2.plot(steps, train_rewards, color='#94a3b8', linewidth=1, alpha=0.5, label='train (batch avg)')
ax2.plot(eval_steps, val_rewards, color='#3b82f6', linewidth=2.5, marker='o', markersize=5, label='val reward')
ax2.axhline(y=train_rewards[0] if train_rewards else 0.3, color='#94a3b8', linestyle='--', alpha=0.5, label='baseline')
ax2.set_title('Episode Reward (mean)', fontweight='bold')
ax2.set_xlabel('Step'); ax2.set_ylabel('Reward (0→1)')
ax2.set_ylim(0, 1.05); ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f8fafc')

# 3. Action match accuracy
ax3.plot(eval_steps, action_match, color='#10b981', linewidth=2.5, marker='s', markersize=5)
ax3.fill_between(eval_steps, action_match, alpha=0.15, color='#10b981')
ax3.set_title('Action Match Accuracy', fontweight='bold')
ax3.set_xlabel('Step'); ax3.set_ylabel('Accuracy (0→1)')
ax3.set_ylim(0, 1.05); ax3.grid(True, alpha=0.3)
ax3.set_facecolor('#f8fafc')
ax3.axhline(y=0.9, color='#059669', linestyle=':', alpha=0.6, label='target 0.90')
ax3.legend(fontsize=8)

# 4. Phishing detection accuracy
ax4.plot(eval_steps, phishing_acc, color='#f59e0b', linewidth=2.5, marker='^', markersize=6)
ax4.fill_between(eval_steps, phishing_acc, alpha=0.15, color='#f59e0b')
ax4.axhline(y=0.95, color='#d97706', linestyle=':', alpha=0.7, label='target 0.95')
ax4.set_title('Phishing Detection Accuracy 🛡️', fontweight='bold')
ax4.set_xlabel('Step'); ax4.set_ylabel('Accuracy (0→1)')
ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)
ax4.set_facecolor('#f8fafc')

plt.savefig('grpo_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved to grpo_training_curves.png')
print('   Upload this file or share your W&B run link with judges.')
if wandb.run and wandb.run.url:
    print(f'   W&B Run: {wandb.run.url}')

In [ ]:
# ── Cell 11: Final evaluation on live OpenEnv environment (optional)
# Requires your HF Space to be running. Skip if offline.
import requests

def run_live_eval(env_url: str, n_episodes: int = 5) -> dict:
    """Run n_episodes against the live OpenEnv environment and report scores."""
    scores = []
    FastLanguageModel.for_inference(model)

    for ep in range(n_episodes):
        try:
            r = requests.post(f'{env_url}/reset', timeout=30)
            obs = r.json()
            task = obs.get('observation', obs).get('task', {})
            ep_rewards = []

            for step in range(10):
                email_ctx = (f"From: {task.get('email_sender','')}\n"
                             f"Subject: {task.get('email_subject','')}\n"
                             f"Body: {task.get('email_body','')}")
                prompt = make_prompt(email_ctx)
                inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                                        pad_token_id=tokenizer.eos_token_id)
                input_len = inputs['input_ids'].shape[1]
                action_str = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

                # Try to parse and clean the action
                parsed = parse_action(action_str)
                if parsed:
                    action_str = json.dumps(parsed)

                step_r = requests.post(f'{env_url}/step', json={'action': action_str}, timeout=30)
                step_obs = step_r.json()
                obs_data = step_obs.get('observation', step_obs)
                reward = float(obs_data.get('reward', 0))
                done   = bool(obs_data.get('done', False) or step_obs.get('done', False))
                ep_rewards.append(reward)

                if done: break

            ep_score = max(ep_rewards) if ep_rewards else 0.001
            scores.append(ep_score)
            print(f'  Episode {ep+1}/{n_episodes}: score={ep_score:.3f}  steps={len(ep_rewards)}')

        except requests.exceptions.RequestException as e:
            print(f'  Episode {ep+1}: connection error ({e}) — skipping')

    FastLanguageModel.for_training(model)
    avg = sum(scores) / len(scores) if scores else 0.0
    return {'scores': scores, 'avg_score': avg, 'n_episodes': len(scores)}

print('Running live evaluation against HF Space...')
live_results = run_live_eval(ENV_URL, n_episodes=5)
print(f'\n📊 Live Env Results:')
print(f'   Avg score: {live_results["avg_score"]:.3f}')
print(f'   Scores:    {[f"{s:.3f}" for s in live_results["scores"]]}')
wandb.log({'live/avg_score': live_results['avg_score']})

In [ ]:
# ── Cell 12: Save final model + push to HuggingFace Hub (optional)
PUSH_TO_HUB = False   # set to True to push
HUB_REPO_ID = 'nishtha711/email-triage-grpo-qwen2.5-0.5b'

# Save locally
final_path = f'{cfg.output_dir}/final'
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f'✅ Final model saved to {final_path}')

if PUSH_TO_HUB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = os.getenv('HF_TOKEN')

    model.push_to_hub(HUB_REPO_ID, token=hf_token)
    tokenizer.push_to_hub(HUB_REPO_ID, token=hf_token)
    print(f'✅ Pushed to HuggingFace Hub: https://huggingface.co/{HUB_REPO_ID}')

# Final summary
print('\n' + '='*60)
print('📊 TRAINING SUMMARY')
print('='*60)
print(f'Model:           {MODEL_ID}')
print(f'GRPO steps:      {cfg.num_iterations}')
print(f'Batch size:      {cfg.batch_size} prompts × G={cfg.group_size}')
print(f'Baseline reward: {history["val_reward"][0]:.3f}')
final_val = next((v for v in reversed(history['val_reward']) if v is not None), 0.0)
print(f'Final val reward:{final_val:.3f}')
print(f'Improvement:     +{final_val - history["val_reward"][0]:.3f}')
print(f'Live env score:  {live_results["avg_score"]:.3f}')
if wandb.run:
    print(f'W&B Run:         {wandb.run.url}')
print('='*60)

wandb.finish()